# 02 — Bronze: Ingestão CSV → DuckDB

Lê os 11 CSVs exportados do Northwind usando `read_csv_auto()` e grava na camada bronze.

**Técnica:** Full Load — DELETE + INSERT por tabela, equivalente ao TRUNCATE + INSERT do SQL Server.

**Diferencial DuckDB:** `read_csv_auto()` detecta schema automaticamente — zero configuração para inferência de tipos.

**Fonte:** `duckdb/data/*.csv` (gerados pelo `export/export_northwind.py`)

**Destino:** `bronze.*` (tabelas DuckDB)

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Tabelas de referência (dimensões)
# read_csv_auto() detecta tipos automaticamente
# ============================================================
print("Ingestão bronze — tabelas de referência:")

conn.execute("DELETE FROM bronze.customers")
conn.execute(f"""
    INSERT INTO bronze.customers
    SELECT *, current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/customers.csv', header=true)
""")
print(f"  bronze.customers:     {conn.execute('SELECT COUNT(*) FROM bronze.customers').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.employees")
conn.execute(f"""
    INSERT INTO bronze.employees
    SELECT
        EmployeeID, LastName, FirstName, Title, TitleOfCourtesy,
        TRY_CAST(BirthDate AS TIMESTAMP) AS BirthDate,
        TRY_CAST(HireDate  AS TIMESTAMP) AS HireDate,
        Address, City, Region, PostalCode, Country, HomePhone, Extension,
        TRY_CAST(ReportsTo AS INTEGER) AS ReportsTo, PhotoPath,
        current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/employees.csv', header=true)
""")
print(f"  bronze.employees:     {conn.execute('SELECT COUNT(*) FROM bronze.employees').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.products")
conn.execute(f"""
    INSERT INTO bronze.products
    SELECT
        ProductID, ProductName, SupplierID, CategoryID, QuantityPerUnit,
        TRY_CAST(UnitPrice     AS DOUBLE)  AS UnitPrice,
        TRY_CAST(UnitsInStock  AS INTEGER) AS UnitsInStock,
        TRY_CAST(UnitsOnOrder  AS INTEGER) AS UnitsOnOrder,
        TRY_CAST(ReorderLevel  AS INTEGER) AS ReorderLevel,
        TRY_CAST(Discontinued  AS BOOLEAN) AS Discontinued,
        current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/products.csv', header=true)
""")
print(f"  bronze.products:      {conn.execute('SELECT COUNT(*) FROM bronze.products').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.categories")
conn.execute(f"""
    INSERT INTO bronze.categories
    SELECT *, current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/categories.csv', header=true)
""")
print(f"  bronze.categories:    {conn.execute('SELECT COUNT(*) FROM bronze.categories').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.suppliers")
conn.execute(f"""
    INSERT INTO bronze.suppliers
    SELECT *, current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/suppliers.csv', header=true)
""")
print(f"  bronze.suppliers:     {conn.execute('SELECT COUNT(*) FROM bronze.suppliers').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.shippers")
conn.execute(f"""
    INSERT INTO bronze.shippers
    SELECT *, current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/shippers.csv', header=true)
""")
print(f"  bronze.shippers:      {conn.execute('SELECT COUNT(*) FROM bronze.shippers').fetchone()[0]} linhas")

Ingestão bronze — tabelas de referência:
  bronze.customers:     91 linhas
  bronze.employees:     9 linhas
  bronze.products:      77 linhas
  bronze.categories:    8 linhas
  bronze.suppliers:     29 linhas
  bronze.shippers:      3 linhas


In [3]:
# ============================================================
# Tabelas transacionais
# ============================================================
print("Ingestão bronze — tabelas transacionais:")

conn.execute("DELETE FROM bronze.orders")
conn.execute(f"""
    INSERT INTO bronze.orders
    SELECT
        OrderID, CustomerID, EmployeeID,
        TRY_CAST(OrderDate    AS TIMESTAMP) AS OrderDate,
        TRY_CAST(RequiredDate AS TIMESTAMP) AS RequiredDate,
        TRY_CAST(ShippedDate  AS TIMESTAMP) AS ShippedDate,
        ShipVia, TRY_CAST(Freight AS DOUBLE) AS Freight,
        ShipName, ShipAddress, ShipCity, ShipRegion, ShipPostalCode, ShipCountry,
        current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/orders.csv', header=true)
""")
print(f"  bronze.orders:        {conn.execute('SELECT COUNT(*) FROM bronze.orders').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.order_details")
conn.execute(f"""
    INSERT INTO bronze.order_details
    SELECT
        TRY_CAST(OrderID   AS INTEGER) AS OrderID,
        TRY_CAST(ProductID AS INTEGER) AS ProductID,
        TRY_CAST(UnitPrice AS DOUBLE)  AS UnitPrice,
        TRY_CAST(Quantity  AS INTEGER) AS Quantity,
        TRY_CAST(Discount  AS DOUBLE)  AS Discount,
        current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/order_details.csv', header=true)
""")
print(f"  bronze.order_details: {conn.execute('SELECT COUNT(*) FROM bronze.order_details').fetchone()[0]} linhas")

Ingestão bronze — tabelas transacionais:


  bronze.orders:        830 linhas
  bronze.order_details: 2155 linhas


In [4]:
# ============================================================
# Tabelas de território
# ============================================================
print("Ingestão bronze — territórios:")

conn.execute("DELETE FROM bronze.territories")
conn.execute(f"""
    INSERT INTO bronze.territories
    SELECT *, current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/territories.csv', header=true)
""")
print(f"  bronze.territories:   {conn.execute('SELECT COUNT(*) FROM bronze.territories').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.region")
conn.execute(f"""
    INSERT INTO bronze.region
    SELECT *, current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/region.csv', header=true)
""")
print(f"  bronze.region:        {conn.execute('SELECT COUNT(*) FROM bronze.region').fetchone()[0]} linhas")

conn.execute("DELETE FROM bronze.employee_territories")
conn.execute(f"""
    INSERT INTO bronze.employee_territories
    SELECT *, current_timestamp AS _LoadTimestamp
    FROM read_csv_auto('{DATA_DIR}/employeeterritories.csv', header=true)
""")
print(f"  bronze.employee_territories: {conn.execute('SELECT COUNT(*) FROM bronze.employee_territories').fetchone()[0]} linhas")

Ingestão bronze — territórios:
  bronze.territories:   53 linhas
  bronze.region:        4 linhas
  bronze.employee_territories: 49 linhas


In [5]:
# ============================================================
# Validação: contagem por tabela
# ============================================================
bronze_tables = [
    ("bronze.customers",           91),
    ("bronze.employees",            9),
    ("bronze.products",            77),
    ("bronze.categories",           8),
    ("bronze.suppliers",           29),
    ("bronze.shippers",             3),
    ("bronze.orders",             830),
    ("bronze.order_details",      2155),
    ("bronze.territories",         53),
    ("bronze.region",               4),
    ("bronze.employee_territories", 49),
]

print("\nValidação bronze:")
print(f"{'Tabela':<45} {'Contagem':>8} {'Esperado':>9} {'OK':>5}")
print("-" * 70)
all_ok = True
for table, expected in bronze_tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    ok = n == expected
    all_ok = all_ok and ok
    print(f"  {table:<43} {n:>8} {expected:>9} {'OK' if ok else 'FAIL':>5}")

print("-" * 70)
print(f"Resultado: {'PASS' if all_ok else 'FAIL'}")


Validação bronze:
Tabela                                        Contagem  Esperado    OK
----------------------------------------------------------------------
  bronze.customers                                  91        91    OK
  bronze.employees                                   9         9    OK
  bronze.products                                   77        77    OK
  bronze.categories                                  8         8    OK
  bronze.suppliers                                  29        29    OK
  bronze.shippers                                    3         3    OK
  bronze.orders                                    830       830    OK
  bronze.order_details                            2155      2155    OK
  bronze.territories                                53        53    OK
  bronze.region                                      4         4    OK
  bronze.employee_territories                       49        49    OK
----------------------------------------------------------

In [6]:
# Preview: primeiras linhas de orders
print("Preview bronze.orders:")
print(conn.execute("""
    SELECT OrderID, CustomerID, EmployeeID, OrderDate::DATE, RequiredDate::DATE,
           ShippedDate::DATE, ShipCountry
    FROM bronze.orders
    LIMIT 5
""").fetchdf().to_string(index=False))

conn.close()

Preview bronze.orders:


 OrderID CustomerID  EmployeeID CAST(OrderDate AS DATE) CAST(RequiredDate AS DATE) CAST(ShippedDate AS DATE) ShipCountry
   10248      VINET           5              1996-07-04                 1996-08-01                1996-07-16      France
   10249      TOMSP           6              1996-07-05                 1996-08-16                1996-07-10     Germany
   10250      HANAR           4              1996-07-08                 1996-08-05                1996-07-12      Brazil
   10251      VICTE           3              1996-07-08                 1996-08-05                1996-07-15      France
   10252      SUPRD           4              1996-07-09                 1996-08-06                1996-07-11     Belgium
